# Clase 1: Agentic Loops y el Ciclo Tool Use

## 1. Qué es un LLM?

Un modelo de lenguaje grande (LLM), es una red neuronal profunda basada en la arquitectura Transformer, entrenada sobre grandes volúmenes de texto mediante aprendizaje auto-supervisado. Su tarea fundamental es modelar la probabilidad del siguiente token en una secuencia: dado un contexto de entrada, el modelo estima cuál es el token más probable que viene a continuación.
Para lograrlo, el texto se divide en tokens —unidades de texto como palabras, subpalabras o caracteres— y cada token se convierte en un vector numérico llamado embedding. Estos vectores forman una matriz que atraviesa el Transformer, donde cálculos sucesivos transforman la información incorporando el contexto completo de la secuencia. Al final, el modelo produce logits: puntuaciones sobre todo el vocabulario. El token con la puntuación más alta es el predicho como siguiente.
Repitiendo esta operación token por token, el modelo genera texto coherente y contextualmente relevante.

![Diagrama Pipeline LLM](img/llm_pipeline.svg)

## 2. Agentic Loops y el Ciclo tool_use

Un agentic loop es el patrón fundamental para construir agentes con Claude. El modelo no solo genera texto: inspecciona resultados, decide qué herramientas usar y repite hasta completar la tarea.

---

### ReAct
**ReAct** (**Reasoning + Acting**) es un patrón para construir agentes con LLMs en el que el modelo **alterna razonamiento y acción** dentro de un mismo bucle, en lugar de razonar todo de golpe o actuar a ciegas.

La idea central, propuesta por Yao et al. (2022) en el paper *"ReAct: Synergizing Reasoning and Acting in Language Models"*, es intercalar tres tipos de pasos en una traza iterativa:

- **Thought (pensamiento):** el modelo razona en lenguaje natural sobre el estado actual y decide qué hacer a continuación.
- **Action (acción):** emite una llamada a una herramienta externa (buscar, consultar una API, ejecutar código, etc.).
- **Observation (observación):** recibe el resultado de esa acción y lo incorpora al contexto.

---

### 2.1 El lifecycle del agentic loop

El flujo básico tiene 4 fases que se repiten:

1. Enviar request a la API (messages + tools disponibles)
2. Inspeccionar `stop_reason` de la respuesta
3. Si `stop_reason` == `tool_use` → ejecutar tools → agregar results → volver a 1
4. Si `stop_reason` == `end_turn` → el agente terminó, entregar respuesta


![Diagrama Pipeline LLM](img/tool_use.png)

La clave es que **el modelo decide** cuándo ha terminado. No es el desarrollador quien programa condiciones de salida — es Claude quien evalúa si la tarea está completa.

### `stop_reason`: la señal de control

| `stop_reason` | Significado | Acción |
| :--- | :--- | :--- |
| **`"tool_use"`** | Claude quiere ejecutar una o más tools | Ejecutar, agregar results, continuar loop |
| **`"end_turn"`** | Claude considera la tarea completa | Salir del loop, entregar respuesta al usuario |
| **`"max_tokens"`** | Se alcanzó el límite de tokens | Manejar como caso especial (truncamiento) |

---

Cuando Claude solicita una herramienta, la respuesta incluye un `tool_use` content block con:

- `id`: identificador único del llamado
- `name`: nombre de la herramienta
- `input`: parámetros que Claude eligió

El desarrollador ejecuta la herramienta y devuelve el resultado como un mensaje con role `"user"` que contiene un `tool_result` content block:  
  
```json
{
  "role": "user",
  "content": [
    {
      "type": "tool_result",
      "tool_use_id": "toolu_abc123",
      "content": "El archivo tiene 42 líneas"
    }
  ]
}
```

Este resultado se agrega al array `messages` completo y se envía de vuelta a la API. Claude ve todo el historial — incluyendo sus llamadas previas y los resultados — para decidir el siguiente paso.

#### Ejemplo: Agentic loop básico en Python SDK Anthropic

In [ ]:
from anthropic import Anthropic
import re

client = Anthropic()
MODEL_NAME = "claude-opus-4-1"

def calculate(expression):
    # Remove any non-digit or non-operator characters from the expression
    expression = re.sub(r"[^0-9+\-*/().]", "", expression)

    try:
        # Evaluate the expression using the built-in eval() function
        # Note: eval is used here for demonstration purposes only.
        # In production, use a safer alternative like ast.literal_eval or a math parser.
        result = eval(expression)  # noqa: S307
        return str(result)
    except (SyntaxError, ZeroDivisionError, NameError, TypeError, OverflowError):
        return "Error: Invalid expression"

tools = [
    {
        "name": "calculator",
        "description": "A simple calculator that performs basic arithmetic operations.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The mathematical expression to evaluate (e.g., '2 + 3 * 4').",
                }
            },
            "required": ["expression"],
        },
    }
]

def process_tool_call(tool_name, tool_input):
    if tool_name == "calculator":
        return calculate(tool_input["expression"])


def chat_with_claude(user_message):
    print(f"\n{'=' * 50}\nUser Message: {user_message}\n{'=' * 50}")

    messages=[{"role": "user", "content": user_message}]

    message = client.messages.create(
            model=MODEL_NAME,
            max_tokens=4096,
            messages=messages,
            tools=tools,
        )

    print("\nInitial Response:")
    print(f"Stop Reason: {message.stop_reason}")
    print(f"Content: {message.content}")

    while message.stop_reason != 'end_turn':

        if message.stop_reason == "tool_use":
            tool_use = next(block for block in message.content if block.type == "tool_use")
            tool_name = tool_use.name
            tool_input = tool_use.input

            print(f"\nTool Used: {tool_name}")
            print(f"Tool Input: {tool_input}")

            tool_result = process_tool_call(tool_name, tool_input)

            print(f"Tool Result: {tool_result}")

            messages.extend(
                [   
                    {
                        "role": "assistant", 
                        "content": message.content
                    },
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "tool_result",
                                "tool_use_id": tool_use.id,
                                "content": tool_result,
                            }
                        ],
                    }
                ]
            )

            message = client.messages.create(
                model=MODEL_NAME,
                max_tokens=4096,
                messages=messages,
                tools=tools,
            )

#       if message.stop_reason == "max_tokens":
#           ... Implementar estrategia para cuando se alcanza el máximo de tokens

    final_response = next(
        (block.text for block in message.content if hasattr(block, "text")),
        None,
    )
    print(message.content)
    print(f"\nFinal Response: {final_response}")

    return final_response

In [10]:
chat_with_claude("What is the result of 1,984,135 * 9,343,116?")


User Message: What is the result of 1,984,135 * 9,343,116?

Initial Response:
Stop Reason: tool_use
Content: [TextBlock(citations=None, text="I'll calculate that multiplication for you.", type='text'), ToolUseBlock(id='toolu_01FF9NeSokqb2LWJy3KDjig2', caller=DirectCaller(type='direct'), input={'expression': '1984135 * 9343116'}, name='calculator', type='tool_use')]

Tool Used: calculator
Tool Input: {'expression': '1984135 * 9343116'}
Tool Result: 18538003464660
[TextBlock(citations=None, text='The result of 1,984,135 × 9,343,116 is **18,538,003,464,660**.', type='text')]

Final Response: The result of 1,984,135 × 9,343,116 is **18,538,003,464,660**.


'The result of 1,984,135 × 9,343,116 is **18,538,003,464,660**.'

#### Ejemplo: Agentic loop básico en Python SDK Claude Agent

In [ ]:
from typing import Any
from claude_agent_sdk import tool, create_sdk_mcp_server

@tool(
    name = "calculate",
    description = "A simple calculator that performs basic arithmetic operations.",
    input_schema={'expression': str}
)
async def calculate(args: dict[str, Any]) -> str:
    
    expression = args['expression']
    
    # Remove any non-digit or non-operator characters from the expression
    expression = re.sub(r"[^0-9+\-*/().]", "", expression)

    try:
        result = eval(expression)  # noqa: S307
        return str(result)
    except (SyntaxError, ZeroDivisionError, NameError, TypeError, OverflowError):
        return "Error: Invalid expression"

math_server = create_sdk_mcp_server(
    name="math",
    version="1.0.0",
    tools=[calculate],
)

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

options = ClaudeAgentOptions(
    mcp_servers={"math": math_server},
    allowed_tools=["mcp__math__calculate"],
)

async for message in query(
    prompt="What is the result of 1,984,135 * 9,343,116?",
    options=options,
):

    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)

The result of **1,984,135 × 9,343,116** is:

## **18,538,003,464,660**


---

### Anti-Patterns del Agentic Loop

##### 1. Parsear lenguaje natural para terminar
```python
# MAL: frágil y no confiable
if (response.content[0].text.includes("tarea completada"))
  break
```

El texto del modelo puede variar. Usar `stop_reason` es determinístico.

##### 2. Caps arbitrarios de iteraciones
```python
# MAL: corta el razonamiento prematuramente
MAX_ITERATIONS = 3
for i in range(MAX_ITERATIONS):
  ...
```

Si Claude necesita 5 iteraciones para completar una tarea, cortarlo en 3 produce resultados incompletos. Si necesitas un safety net, ponlo alto (ej: 50) y loguéalo como anomalía.

##### 3. Buscar texto del asistente como indicador
```python
# MAL: el modelo puede generar texto Y tool calls juntos
if any(b.type == "text" for b in response.content):
    return response  # podría haber tool_use blocks también
```

Una respuesta puede contener tanto text blocks como `tool_use` blocks simultáneamente. Solo `stop_reason` indica la intención real.

##### 4. Ignorar `tool_use` blocks múltiples
Claude puede solicitar varias herramientas en un solo turno. Ejecutar solo la primera y descartar las demás produce información incompleta para el siguiente razonamiento.

## 3. SubAgentes - Patrones Coordinator

El patrón coordinator-subagent es la arquitectura principal para sistemas multi-agente con Claude. Un agente coordinador descompone tareas complejas y delega a agentes especializados.

---

### Rol del coordinator

El coordinator tiene 4 responsabilidades principales:

##### 1. Descomposición de tareas
Analiza la tarea del usuario y la divide en subtareas manejables. Cada subtarea debe ser lo suficientemente específica para que un subagente la complete sin ambigüedad.  

Tarea: "Refactoriza el módulo de autenticación".  
  ├── Subtarea 1: Analizar el código actual y sus dependencias.  
  ├── Subtarea 2: Diseñar la nueva estructura.  
  ├── Subtarea 3: Implementar los cambios.  
  └── Subtarea 4: Verificar que los tests pasan.  

##### 2. Delegación
Asigna cada subtarea al subagente más apropiado según su especialización y las herramientas disponibles.

##### 3. Aggregación de resultados
Recopila los outputs de todos los subagentes, resuelve conflictos y compone una respuesta coherente.

##### 4. Routing por complejidad
No todas las tareas requieren subagentes. El coordinator decide dinámicamente:

| Complejidad | Acción |
| :--- | :--- |
| **Simple** (una sola operación) | El coordinator la resuelve directamente |
| **Media** (2-3 pasos conocidos) | Delega a un subagente |
| **Alta** (múltiples pasos, incertidumbre) | Delega a varios subagentes |

---

### Contexto aislado de los subagentes
Un subagente **no hereda el historial de conversación del coordinator**. Cada subagente empieza con un contexto limpio que incluye:

- Su system prompt específico
- El prompt de la tarea asignada por el coordinator
- Las herramientas que tiene disponibles

Esto es una decisión de diseño deliberada:

- **Reduce contaminación:** el subagente no se distrae con contexto irrelevante
- **Reduce tokens:** no consume context window con historial ajeno
- **Mejora enfoque:** el subagente solo ve lo necesario para su tarea

Si un subagente necesita información del contexto del coordinator, este debe incluirla explícitamente en el prompt de la tarea.

---

### Routing dinámico vs pipeline fijo

#### Pipeline fijo
Todas las tareas pasan por todos los subagentes en secuencia:  

Análisis → Diseño → Implementación → Testing.  
Útil cuando el proceso siempre es el mismo. Desperdicia recursos si una tarea solo necesita uno o dos pasos.  

#### Routing dinámico
El coordinator evalúa cada tarea y decide qué subagentes invocar:

`Corrige el typo en el README.`  
  → Solo invocar al editor (no necesita análisis ni tests).  

`Agrega un nuevo endpoint con validación.`  
  → Análisis + Implementación + Testing. 

El routing dinámico es más eficiente pero requiere que el coordinator tenga buen juicio. Para lograrlo, su system prompt debe incluir criterios claros de routing, no dejar la decisión totalmente abierta.




## 4. Subagentes — Invocación, Contexto y Spawning

Los subagentes son instancias de Claude que un agente coordinador puede invocar para delegar tareas específicas. Operan con contexto aislado y reciben sus instrucciones explícitamente desde el coordinador.

### La herramienta Task como mecanismo de spawning

El spawning de subagentes se hace mediante la herramienta especial `Task` del Agent SDK. Para que un coordinador pueda invocar subagentes, su configuración `allowedTools` debe incluir explícitamente `Task`:

```python
from claude_agent_sdk import ClaudeAgentOptions

ClaudeAgentOptions(
    system_prompt = "Coordina la investigación delegando a subagentes...",
    allowed_tools = ["Task", "WebSearch"],
)
```

Sin `Task` en `allowed_tools`, el coordinador no puede emitir llamadas a subagentes — la herramienta no es visible para el modelo.

---

### AgentDefinition: especialización por subagente

Cada tipo de subagente se define con una AgentDefinition que incluye:

- **Description**: cuándo usar este subagente (vital para que el coordinador lo seleccione bien).
- **System prompt**: rol específico, tono, criterios de salida.
- **Tool restrictions**: solo las herramientas relevantes a su rol.

---

### Spawning paralelo
Para invocar múltiples subagentes en paralelo, el coordinador emite **varias llamadas a `Task` en una sola respuesta**, no en turnos separados:

```python
# Una sola respuesta del coordinador con 3 tool_use blocks
[
  { type: "tool_use", name: "Task", input: { agent: "search", prompt: "..." }},
  { type: "tool_use", name: "Task", input: { agent: "search", prompt: "..." }},
  { type: "tool_use", name: "Task", input: { agent: "analyze", prompt: "..." }},
]
```

Esto permite que los subagentes trabajen simultáneamente, reduciendo la latencia total.

---

In [4]:
import asyncio
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AgentDefinition,
    AssistantMessage,
    ResultMessage,
    TextBlock,
)

# --- Los dos subagentes especializados ---

researcher = AgentDefinition(
    description="Busca en internet información actualizada sobre un tema. Úsalo cuando se necesiten datos, fuentes o hechos.",
    model="claude-sonnet-4-6",
    prompt=(
        "Sos un agente investigador. Buscás en la web información relevante y "
        "confiable sobre el tema que te asignen. Devolvés hallazgos concretos, "
        "cada uno con su fuente (URL). No redactás el texto final, solo recopilás."
    ),
    tools=["WebSearch"],   # solo la herramienta que necesita
)

writer = AgentDefinition(
    description="Redacta un texto claro y bien estructurado a partir de hallazgos ya recopilados.",
    model="claude-sonnet-4-6",
    prompt=(
        "Sos un agente redactor. Recibís hallazgos con sus fuentes y producís "
        "un texto coherente, claro y bien organizado. No buscás información nueva: "
        "solo trabajás con lo que el coordinador te pasa en el prompt."
    ),
    tools=[],   # no necesita herramientas, solo escribe
)

# --- El agente líder (coordinador) ---

options = ClaudeAgentOptions(
    model="claude-sonnet-4-6",
    system_prompt=(
        "Sos el investigador líder. Tu trabajo es coordinar, no ejecutar todo vos mismo.\n"
        "1. Delegá la búsqueda de información al subagente 'researcher'.\n"
        "2. Pasale esos hallazgos (con sus fuentes) al subagente 'writer' para que redacte.\n"
        "3. Devolvé al usuario el texto final redactado."
    ),
    agents={
        "researcher": researcher,
        "writer": writer,
    },
    allowed_tools=["Task"],   # 'Task' = mecanismo para spawnear subagentes
)

In [5]:
# --- Ejecutar el loop ---
async for message in query(
    prompt="Investigá y redactá un resumen sobre el impacto de los LLM en la educación.",
    options=options,
):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print(block.text)
    elif isinstance(message, ResultMessage) and message.subtype == "success":
        print("\n=== RESULTADO FINAL ===")
        print(message.result)

Excelente. Ahora le paso todos los hallazgos al subagente **writer** para que redacte el informe final.
Aquí está el informe final, redactado a partir de los hallazgos de investigación:

---

# Modelos de Lenguaje Grandes en Educación: Impacto, Riesgos y Perspectivas

## Introducción

La irrupción de los modelos de lenguaje grandes (LLMs) en el ámbito educativo constituye uno de los fenómenos tecnológicos más significativos de los últimos años. Sistemas como ChatGPT, Gemini o Copilot han pasado de ser herramientas experimentales a recursos de uso cotidiano entre estudiantes y docentes de todos los niveles y regiones del mundo. La velocidad de esta adopción es, en sí misma, un dato revelador: según el Pew Research Center (2024), el **26% de los adolescentes estadounidenses de entre 13 y 17 años** ya utiliza ChatGPT para tareas escolares, cifra que se duplicó en apenas un año; entre los jóvenes de 18 a 29 años, ese porcentaje asciende al 43%. En el plano institucional, el **69% de los lí

## 5. Workflows Multi-step, Hooks y Handoffs

Los workflows reales requieren más que delegación de tareas. Necesitan enforcement de reglas, normalización de datos, prerequisitos y transferencias estructuradas entre agentes o humanos.

### Enforcement programático vs guidance via prompt

Hay dos formas de controlar el comportamiento de un agente:

#### 1. Guidance via prompt
Instrucciones en el system prompt o tarea que Claude debería seguir:  

**System prompt:** `Nunca proceses un reembolso mayor a $500 sin aprobación.`  

Claude seguirá esta instrucción la mayoría de las veces, pero no hay garantía determinística. En escenarios de alto riesgo, guidance no es suficiente

#### 2. Enforcement programático
Código que impide que una acción ocurra sin cumplir condiciones:  


```python
async def process_refund(amount: float, approved: bool):
  if (amount > 500 and not approved):
    raise("Refunds > $500 require manager approval")
  # procesar...
```

#### Cuándo usar cada uno

| Escenario | Mecanismo |
| :--- | :--- |
| Formato de respuesta preferido | Guidance (prompt) |
| Tono y estilo de comunicación | Guidance (prompt) |
| Verificación de identidad antes de operaciones financieras | Enforcement (código) |
| Límites de reembolso | Enforcement (código) |
| Bloqueo de acciones destructivas | Enforcement (código) |
| Orden sugerido de pasos | Guidance (prompt) |


**Regla:** si el incumplimiento causa daño material (dinero, datos, seguridad), usa enforcement programático. Si es una preferencia, usa guidance.

---

### Hooks

Los hooks son funciones de devolución de llamada que ejecutan su código en respuesta a eventos del agente, como una herramienta siendo llamada, una sesión iniciándose, o la ejecución deteniéndose. 

#### PostToolUse

Los hooks `PostToolUse` interceptan el resultado de una herramienta **después** de su ejecución y **antes** de que Claude lo procese. Son ideales para normalizar datos.

##### Normalización de timestamps
Una herramienta externa puede devolver timestamps en formatos inconsistentes:  

API A: "1710547200"        (Unix epoch)  
API B: "2024-03-16T00:00Z" (ISO 8601)  
API C: "March 16, 2024"    (texto)  

Un PostToolUse hook normaliza todo a un formato consistente.

`PostToolUse` no puede bloquear nada (la herramienta ya se ejecutó); solo observa o transforma. Si en algún ejemplo quisieras validar/denegar antes de llamar a la API de pagos, ese sí sería el caso de uso natural de `PreToolUse` con permissionDecision: "deny".

#### PreToolUse

Los hooks pueden interceptar antes de ejecutar una herramienta `PreToolUse` para bloquear acciones que violan políticas.

In [14]:
from datetime import datetime, timezone
from typing import Any

from claude_agent_sdk import (
    tool,
    create_sdk_mcp_server,
    query,
    ClaudeAgentOptions,
    HookMatcher,
    HookContext,
    ResultMessage,
)


# --- 1. Una herramienta que "trae" una fecha en un formato crudo (Unix epoch) ---
@tool(
    name="obtener_fecha",
    description="Devuelve la fecha/hora actual como timestamp Unix (segundos).",
    input_schema={},
)
async def obtener_fecha(args: dict[str, Any]) -> dict[str, Any]:
    epoch = int(datetime.now(timezone.utc).timestamp())
    # La herramienta devuelve un formato poco amigable a propósito
    return {"content": [{"type": "text", "text": str(epoch)}]}


fechas_server = create_sdk_mcp_server(
    name="fechas",
    version="1.0.0",
    tools=[obtener_fecha],
)


def _extraer_texto(tool_response: Any) -> str:
    """El SDK puede entregar tool_response como str, lista de blocks o dict con 'content'."""
    if isinstance(tool_response, str):
        return tool_response
    if isinstance(tool_response, dict):
        tool_response = tool_response.get("content", tool_response)
    if isinstance(tool_response, list):
        for block in tool_response:
            if isinstance(block, dict) and block.get("type") == "text":
                return block["text"]
    return str(tool_response)


# --- 2. El hook PostToolUse: se ejecuta DESPUÉS de llamar la herramienta ---
# Firma estándar del SDK: (input_data, tool_use_id, context)
async def normalizar_fecha(
    input_data: dict[str, Any],
    tool_use_id: str | None,
    context: HookContext,
) -> dict[str, Any]:
    # Solo nos interesa nuestra herramienta de fechas
    if input_data["tool_name"] != "mcp__fechas__obtener_fecha":
        return {}

    # tool_response es lo que devolvió la herramienta, antes de que Claude lo lea
    epoch_str = _extraer_texto(input_data["tool_response"]).strip()
    legible = datetime.fromtimestamp(int(epoch_str), tz=timezone.utc).strftime(
        "%Y-%m-%d %H:%M:%S UTC"
    )

    print(f"[HOOK PostToolUse] epoch crudo: {epoch_str}  ->  normalizado: {legible}")

    # Inyectamos la versión normalizada como contexto adicional para Claude
    return {
        "hookSpecificOutput": {
            "hookEventName": "PostToolUse",
            "additionalContext": f"La fecha normalizada es: {legible}",
        }
    }


options = ClaudeAgentOptions(
    model="claude-sonnet-4-6",
    mcp_servers={"fechas": fechas_server},
    allowed_tools=["mcp__fechas__obtener_fecha"],
    hooks={
        "PostToolUse": [
            HookMatcher(matcher="mcp__fechas__obtener_fecha", hooks=[normalizar_fecha]),
        ],
    },
)

In [15]:
# --- 3. Ejecutar el loop ---
async for message in query(
    prompt="¿Qué fecha y hora es ahora? Respondé en lenguaje natural.",
    options=options,
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print("\n=== RESULTADO FINAL ===")
        print(message.result)

[HOOK PostToolUse] epoch crudo: 1781555404  ->  normalizado: 2026-06-15 20:30:04 UTC

=== RESULTADO FINAL ===
Ahora mismo son las **20:30 del 15 de junio de 2026** (hora UTC).


---

### Structured handoff summaries

Cuando un agente transfiere una conversación a otro agente o a un humano, el handoff summary debe ser estructurado para evitar pérdida de información:

```python
from typing import Literal
from pydantic import BaseModel, Field

class HandoffSummary(BaseModel):
    customer_id: str
    root_cause: str
    actions_taken: list[str]
    refund_amount: float
    recommended_action: str
    conversation_sentiment: Literal["positive", "neutral", "frustrated"]
    unresolved_issues: list[str]
```

##### Ejemplo concreto

```json
{
  "customer_id": "CUST-12345",
  "root_cause": "Producto defectuoso recibido (modelo XYZ-100)",
  "actions_taken": [
    "Verificada identidad del cliente",
    "Confirmado pedido #ORD-98765",
    "Revisadas fotos del producto dañado"
  ],
  "refund_amount": 750,
  "recommended_action": "Aprobación de manager para refund > $500",
  "conversation_sentiment": "frustrated",
  "unresolved_issues": [
    "Cliente también solicita envío gratuito en próximo pedido"
  ]
}
```

#### Por qué estructurado
- El siguiente agente/humano no necesita leer toda la conversación
- Campos obligatorios aseguran que no se omita información crítica
- Fácil de parsear programáticamente para routing automático
- Auditable: cada handoff queda registrado con datos concretos

---